In [1]:
# Importing the necessary libraries
import transformers  # Hugging Face library for working with transformer models
import torch         # PyTorch library for tensor operations and model handling
# Importing the necessary libraries
from transformers import AutoTokenizer, AutoModelForCausalLM  # For tokenizer and model handling


In [3]:
# Dictionary mapping medical question categories to tailored system prompts
category_prompts = {
    "Basic Medical Knowledge": (
        "You are an experienced medical professional specializing in general medical knowledge. "
        "Provide detailed and comprehensible answers to basic medical questions, ensuring accuracy and clarity."
    ),
    "Pharmacological Queries": (
        "You are a pharmacology expert with extensive knowledge of drugs, their mechanisms, and side effects. "
        "Provide precise and evidence-based information about medications and treatments."
    ),
    "Diagnostic Reasoning": (
        "You are a diagnostic expert proficient in identifying medical conditions. "
        "Offer clear and methodical guidance for diagnostic reasoning and differential diagnoses."
    ),
    "Treatment & Management": (
        "You are a specialist in medical treatment and patient management strategies. "
        "Provide comprehensive explanations of treatment guidelines and management protocols."
    ),
    "Specialized Medical Topics": (
        "You are an expert in advanced medical research and innovations. "
        "Discuss specialized medical topics with a focus on the latest scientific advances."
    ),
    "Preventive Medicine": (
        "You are a preventive medicine specialist with expertise in lifestyle modifications and public health. "
        "Explain preventive strategies and their importance in reducing disease risk."
    ),
    "Specialized Medical Scenarios": (
        "You are a clinical specialist adept at handling unique and complex medical scenarios. "
        "Provide nuanced insights into complications and specialized conditions."
    )
}

# Default system prompt for cases where no category is selected
default_prompt = (
    "You are an expert and experienced medical professional with extensive medical knowledge. "
    "Provide precise, evidence-based medical explanations that are scientifically accurate and comprehensible to a general audience."
)

# Predefined medical questions categorized under different topics
medical_questions = {
    "Basic Medical Knowledge": [
        "What are the primary symptoms of type 2 diabetes?",
        "Explain the pathophysiology of hypertension.",
        "What are the recommended screening protocols for breast cancer?"
    ],
    "Pharmacological Queries": [
        "What are the potential side effects of statins?",
        "How do ACE inhibitors work to manage blood pressure?",
        "Compare the mechanisms of different antidepressant classes."
    ],
    "Diagnostic Reasoning": [
        "What diagnostic tests would you recommend for suspected rheumatoid arthritis?",
        "Describe the differential diagnosis for chest pain in a 45-year-old male."
    ],
    "Treatment & Management": [
        "What are current guidelines for managing type 1 diabetes in adolescents?",
        "Explain the stages of cancer treatment and potential therapies."
    ],
    "Specialized Medical Topics": [
        "How does CRISPR technology potentially impact genetic disease treatment?",
        "What are the latest advances in immunotherapy for cancer?"
    ],
    "Preventive Medicine": [
        "What lifestyle modifications can reduce the risk of cardiovascular disease?",
        "Discuss the importance of vaccination in preventing infectious diseases."
    ],
    "Specialized Medical Scenarios": [
        "What are the complications of untreated gestational diabetes?",
        "Explain the neurological manifestations of multiple sclerosis."
    ]
}

In [4]:
# Import necessary libraries
#import transformers  # Hugging Face library for transformer models
#import torch         # PyTorch library for model execution and tensor handling

class MedicalChatbotPipeline:
    """
    A class-based implementation for interacting with the Hugging Face Transformers pipeline.
    This chatbot is tailored for the HPAI-BSC/Llama3.1-Aloe-Beta-70B medical language model.
    """

    def __init__(self, model_id="HPAI-BSC/Llama3.1-Aloe-Beta-70B"):
        """
        Initializes the MedicalChatbotPipeline class with the specified model.

        Args:
            model_id (str): The identifier of the Hugging Face model to load.
        """
        self.model_id = model_id  # Store the model ID
        
        print("\n[INFO] Initializing chatbot pipeline...")  # Status update

        # Load the model using the pipeline API with optimized settings
        self.pipeline = transformers.pipeline(
            "text-generation",  # Defines the task as text generation
            model=self.model_id,  
            model_kwargs={"torch_dtype": torch.bfloat16},  # Use lower precision for efficiency
            device_map="auto",  # Automatically selects CPU or GPU
        )

    def generate_response(self, system_prompt, user_message, max_tokens=256, temperature=0.6, top_p=0.9):
        """
        Generates a chatbot response based on a system prompt and user query.

        Args:
            system_prompt (str): The system's instructional message to set context.
            user_message (str): The actual question or input from the user.
            max_tokens (int): Maximum tokens allowed in the generated response.
            temperature (float): Controls response randomness (higher = more random).
            top_p (float): Controls response diversity using nucleus sampling.

        Returns:
            str: The chatbot's generated response.
        """

        print("\n[INFO] Formatting input messages...")  # Status update

        # Construct the conversation structure with system and user messages
        messages = [
            {"role": "system", "content": system_prompt},  # System prompt setting chatbot context
            {"role": "user", "content": user_message},  # User's medical question
        ]

        # Format messages into a proper chat template for the model
        prompt = self.pipeline.tokenizer.apply_chat_template(
            messages, 
            tokenize=False,  # Keep input as raw text (no tokenization)
            add_generation_prompt=True  # Add necessary formatting to signal response start
        )

        print("\n[INFO] Defining termination tokens...")  # Status update

        # Define end-of-response (termination) tokens
        terminators = [
            self.pipeline.tokenizer.eos_token_id,  # Standard end-of-sequence token
            self.pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>"),  # Custom model token
        ]

        print("\n[INFO] Generating chatbot response...")  # Status update

        # Generate a response from the model
        outputs = self.pipeline(
            prompt,  # Formatted input
            max_new_tokens=max_tokens,  # Response length limit
            eos_token_id=terminators,  # Define stopping conditions
            do_sample=True,  # Enable sampling for varied responses
            temperature=temperature,  # Controls randomness
            top_p=top_p,  # Enables nucleus sampling for diverse outputs
        )

        # Extract and clean the generated response
        response = outputs[0]["generated_text"][len(prompt):].strip()

        print("\n💬 **Chatbot Response:**")
        print(response)  # Display response in console
        
        return response  # Return response for external use


In [5]:
#AutoModelForCausalLM

# Importing the necessary libraries
#from transformers import AutoTokenizer, AutoModelForCausalLM  # For tokenizer and model handling
#import torch  # PyTorch library for tensor manipulation and device management

class MedicalChatbotAutoModel:
    """
    A class-based implementation for interacting with the Hugging Face AutoModelForCausalLM approach.
    This chatbot is tailored for the HPAI-BSC/Llama3.1-Aloe-Beta-70B medical language model.
    """

    def __init__(self, model_id="HPAI-BSC/Llama3.1-Aloe-Beta-70B"):
        """
        Initializes the MedicalChatbotAutoModel class with the specified model.

        Args:
            model_id (str): The identifier of the Hugging Face model to load.
        """
        # Define the model ID to be used
        self.model_id = model_id

        # Load the tokenizer associated with the model
        # The tokenizer is responsible for converting text to tokens and vice versa
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)

        # Load the model with specified configurations
        # 'torch_dtype=torch.bfloat16' reduces memory usage with lower precision
        # 'device_map="auto"' automatically selects the best available device (CPU/GPU)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_id,
            torch_dtype=torch.bfloat16,  # Optimized precision for efficiency
            device_map="auto",           # Automatically assign the device (CPU/GPU)
        )

    def generate_response(self, user_message, max_tokens=256, temperature=0.6, top_p=0.9):
        """
        Generates a response from the medical chatbot.

        Args:
        user_message (str): The user's input query.
        max_tokens (int): Maximum number of tokens to generate in the response.
        temperature (float): Controls the randomness of the response.
        top_p (float): Controls the diversity of the response using nucleus sampling.

        Returns:
        str: The generated response from the chatbot.
        """
        # Define the structured conversation messages
        messages = [
            {
                "role": "system",
                "content": (
                    "You are an expert medical assistant named Aloe, developed by the High "
                    "Performance Artificial Intelligence Group at Barcelona Supercomputing Center (BSC). "
                    "You are to be a helpful, respectful, and honest assistant."
                ),
            },
            {"role": "user", "content": user_message},
        ]

        # Convert the conversation into input token IDs
        input_ids = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(self.model.device)

        # Define termination tokens
        terminators = [
            self.tokenizer.eos_token_id,  # Standard EOS token
            self.tokenizer.convert_tokens_to_ids("<|eot_id|>"),  # Custom end token
        ]

        # Generate the response
        outputs = self.model.generate(
            input_ids,
            max_new_tokens=max_tokens,  # Apply the specified max_tokens
            eos_token_id=terminators,  # Termination criteria
            do_sample=True,
            temperature=temperature,  # Apply temperature parameter
            top_p=top_p  # Apply top-p nucleus sampling
        )

        # Extract only the newly generated response
        response_ids = outputs[0][input_ids.shape[-1]:]  # Remove input prompt tokens
        response = self.tokenizer.decode(
            response_ids, skip_special_tokens=True
        ).strip()  # Clean response text

        return response




# Comparison

| Feature                     | Transformers Pipeline Approach         | AutoModelForCausalLM Approach            |
|-----------------------------|-----------------------------------------|------------------------------------------|
| **Ease of Use**             | High (Beginner-friendly)               | Moderate to Low (Requires more effort)   |
| **Customization**           | Limited                                | High                                     |
| **Performance Optimization**| Slight Overhead                        | Slightly Faster                          |
| **Flexibility**             | Moderate (Pre-defined functionality)   | High (Full control over model usage)     |
| **Learning Curve**          | Low                                    | High                                     |


In [6]:
# Import the necessary libraries and classes from separate files for each approach.
#from approach1_pipeline import MedicalChatbotPipeline  # Import the pipeline-based chatbot class
#from approach2_automodel import MedicalChatbotAutoModel  # Import the automodel-based chatbot class


def get_user_query():
    """
    Allows the user to select a medical category, choose from related questions, or enter a custom query.
    Returns the selected query and its associated category prompt.
    """
    try:
        # Display category options
        print("\nAvailable medical categories:")
        categories = list(category_prompts.keys())
        for i, category in enumerate(categories, start=1):
            print(f"{i}. {category}")
        print(f"{len(categories) + 1}. Custom Question (No specific category)")

        # Get category selection with improved error handling
        while True:
            try:
                category_choice = int(input("\nEnter the number of your choice: ").strip())
                
                # Handle custom question option
                if category_choice == len(categories) + 1:
                    user_query = input("\nEnter your medical question: ").strip()
                    return user_query, default_prompt
                
                # Handle category selection
                if 1 <= category_choice <= len(categories):
                    selected_category = categories[category_choice - 1]
                    break
                    
                print("Invalid number. Please select from the available options.")
            except ValueError:
                print("Please enter a valid number.")

        # Display questions for selected category
        print(f"\nQuestions for {selected_category}:")
        questions = medical_questions[selected_category]
        for i, question in enumerate(questions, start=1):
            print(f"{i}. {question}")
        print(f"{len(questions) + 1}. Custom question for this category")

        # Get question w/ error handling
        while True:
            try:
                question_choice = int(input("\nSelect a question number: ").strip())
                
                # Handle custom question option
                if question_choice == len(questions) + 1:
                    user_query = input("\nEnter your medical question: ").strip()
                    return user_query, category_prompts[selected_category]
                
                # Handle predefined question selection
                if 1 <= question_choice <= len(questions):
                    return questions[question_choice - 1], category_prompts[selected_category]
                    
                print("Invalid number. Please select from the available options.")
            except ValueError:
                print("Please enter a valid number.")

    except Exception as e:
        print(f"An error occurred: {str(e)}")
        return None, None

In [7]:

def main_pipeline():
    """
    Runs the MedicalChatbotPipeline approach with a user-selected category and query.
    """
    print("\nRunning MedicalChatbotPipeline approach...\n")  # Indicate the approach being used

    # Get the user's chosen medical query
    user_query, system_prompt = get_user_query()

    # Initialize the chatbot using the pipeline approach
    chatbot = MedicalChatbotPipeline()

    # Generate a response using the chatbot
    response = chatbot.generate_response(user_query, system_prompt)

    # Display the chatbot's response
    print("\nResponse (Pipeline Approach):", response)


In [8]:

def main_automodel():
    """
    Runs the MedicalChatbotAutoModel approach with a user-selected category and query.
    """
    print("\nRunning MedicalChatbotAutoModel approach...\n")  # Indicate the approach being used

    # Get the user's chosen medical query
    user_query,system_prompt = get_user_query()

    # Initialize the chatbot using the automodel approach
    chatbot = MedicalChatbotAutoModel()
    
    # Generate a response using the chatbot
    response = chatbot.generate_response(user_query,system_prompt)

    # Display the chatbot's response
    print("\nResponse (AutoModel Approach):", response)



In [ ]:
if __name__ == "__main__":
    """
    Main entry point of the script. Allows the user to:
    1. Select the chatbot approach (Pipeline or AutoModel)
    2. Choose a medical question category
    3. Pick a predefined or custom question
    4. Initialize the chatbot with the selected settings
    """

    # Step 1: Ask the user which chatbot approach they want to use  
    print("\nSelect the chatbot approach to run:")
    print("1. MedicalChatbotPipeline (Pipeline Approach)")
    print("2. MedicalChatbotAutoModel (AutoModel Approach)")
    

    while True:
        choice = input("\nEnter 1 or 2: ").strip()  # User selects chatbot approach  

        if choice == "1":
            chatbot_function = main_pipeline  # Assign function reference  
            break
        elif choice == "2":
            chatbot_function = main_automodel  # Assign function reference  
            break
        else:
            print("Invalid choice. Please enter 1 or 2.")  # Error handling  

    # Step 2: Get the user’s query and associated category prompt using the predefined function  
    user_query, system_prompt = get_user_query()

    # Step 3: Initialize the selected chatbot approach with the chosen settings  
    print("\nInitializing the chatbot with the following parameters:")
    print(f"System Prompt: {system_prompt if system_prompt else 'Custom Input'}")  # Indicate if a category prompt is used  
    print(f"User Question: {user_query}")

    # Call the selected chatbot function (Pipeline or AutoModel) with system prompt and question  
    chatbot_function()


In [ ]:
if __name__ == "__main__":
    """
    Main entry point of the script with improved flow control.
    Streamlines the process of selecting approach and getting user input.
    """
    # Step 1: Select chatbot approach
    print("\nSelect the chatbot approach to run:")
    print("1. MedicalChatbotPipeline (Pipeline Approach)")
    print("2. MedicalChatbotAutoModel (AutoModel Approach)")

    # Initialize variables
    chatbot = None
    
    # Get valid approach selection
    while True:
        choice = input("\nEnter 1 or 2: ").strip()
        
        if choice == "1":
            print("\nInitializing Pipeline approach...")
            chatbot = MedicalChatbotPipeline()
            break
        elif choice == "2":
            print("\nInitializing AutoModel approach...")
            chatbot = MedicalChatbotAutoModel()
            break
        else:
            print("Invalid choice. Please enter 1 or 2.")

    # Step 2: Get user query and generate response
    while True:
        try:
            # Get user query and system prompt
            user_query, system_prompt = get_user_query()
            
            # Use default prompt if none selected
            if system_prompt is None:
                system_prompt = default_prompt
            
            # Generate and display response
            print("\nGenerating response...")
            response = chatbot.generate_response(system_prompt, user_query)
            print("\nResponse:", response)
            
            # Ask if user wants to continue
            continue_chat = input("\nWould you like to ask another question? (y/n): ").lower()
            if continue_chat != 'y':
                break
                
        except Exception as e:
            print(f"\nAn error occurred: {str(e)}")
            print("Please try again.")